# Experiment 12 — Real-World Attacks & Countermeasures
ARP Spoofing · DNS Cache Poisoning · SQL Injection

## Attack 1 — ARP Spoofing

In [ ]:
class ARPCache:
    def __init__(self,name): self.name=name; self.cache={}
    def update(self,ip,mac,src=""):
        old=self.cache.get(ip); self.cache[ip]=mac
        tag=f" (was {old})" if old and old!=mac else ""
        print(f"  [{self.name}] {ip} → {mac}{tag}  ← {src}")
    def get(self,ip): return self.cache.get(ip)

gw_ip="192.168.1.1";  gw_mac="AA:BB:CC:DD:EE:FF"
vt_ip="192.168.1.100"; vt_mac="11:22:33:44:55:66"
at_ip="192.168.1.200"; at_mac="DE:AD:BE:EF:00:01"

victim=ARPCache("Victim"); gateway=ARPCache("Gateway")

print("--- Legitimate ARP replies ---")
victim.update(gw_ip, gw_mac, "real gateway")
gateway.update(vt_ip, vt_mac, "real victim")

print("\n--- Attacker sends gratuitous ARP (poisoning) ---")
victim.update(gw_ip,  at_mac, "ATTACKER (spoofed!)")
gateway.update(vt_ip, at_mac, "ATTACKER (spoofed!)")

print("\n--- MITM established ---")
print(f"Victim→gateway resolves to {victim.get(gw_ip)} (ATTACKER!)")
print(f"Gateway→victim resolves to {gateway.get(vt_ip)} (ATTACKER!)")
print()
print("Countermeasures:")
for c in ["Static ARP entries","Dynamic ARP Inspection (DAI)","arpwatch monitoring","VLAN segmentation","TLS/VPN encryption"]:
    print(f"  ✓ {c}")


## Attack 2 — DNS Cache Poisoning

In [ ]:
class DNSResolver:
    def __init__(self,name,dnssec=False): self.name=name; self.cache={}; self.dnssec=dnssec
    def cache_resp(self,domain,ip,sig=True):
        self.cache[domain]=(ip,sig)
        print(f"  [{self.name}] Cached: {domain} → {ip}  {'[DNSSEC✓]' if sig else '[NO SIG ⚠]'}")
    def query(self,domain):
        if domain not in self.cache: return None
        ip,sig=self.cache[domain]
        if self.dnssec and not sig:
            print(f"  [{self.name}] DNSSEC validation FAILED → BLOCKED"); return None
        return ip

domain="bank.example.com"; real_ip="93.184.216.34"; fake_ip="10.0.0.99"

print("--- Vulnerable resolver (no DNSSEC) ---")
r1=DNSResolver("Resolver-A")
r1.cache_resp(domain, fake_ip, sig=False)
print(f"Client resolves {domain} → {r1.query(domain)}  ⚠ ATTACKER's server!")

print("\n--- Protected resolver (DNSSEC enabled) ---")
r2=DNSResolver("Resolver-B",dnssec=True)
r2.cache_resp(domain, fake_ip, sig=False)
print(f"Attacker response  → {r2.query(domain)}")
r2.cache_resp(domain, real_ip, sig=True)
print(f"Legitimate response → {r2.query(domain)}  ✓")

print()
print("Countermeasures:")
for c in ["DNSSEC (cryptographic signatures)","DNS-over-HTTPS (DoH)","DNS-over-TLS (DoT)","Source-port randomisation","0x20 mixed-case encoding"]:
    print(f"  ✓ {c}")


## Attack 3 — SQL Injection

In [ ]:
import sqlite3, hashlib, os, tempfile

db=tempfile.mktemp(suffix=".db")
conn=sqlite3.connect(db)
conn.execute("CREATE TABLE users(id INTEGER PRIMARY KEY,username TEXT,password_hash TEXT,role TEXT)")
conn.execute("CREATE TABLE secret_data(id INTEGER PRIMARY KEY,data TEXT)")
for u,p,r in [("alice",b"alice123","admin"),("bob",b"bobpass","user")]:
    conn.execute("INSERT INTO users VALUES(?,?,?,?)",(None,u,hashlib.sha256(p).hexdigest(),r))
conn.execute("INSERT INTO secret_data VALUES(?,?)",(None,"TOP SECRET: salary=$200000"))
conn.commit()

def vuln_login(user,pw):
    q=f"SELECT * FROM users WHERE username='{user}' AND password_hash='{pw}'"
    try: rows=conn.execute(q).fetchall(); return rows or "Login failed"
    except Exception as e: return f"Error: {e}"

def safe_login(user,pw):
    ph=hashlib.sha256(pw.encode()).hexdigest()
    rows=conn.execute("SELECT id,username,role FROM users WHERE username=? AND password_hash=?",(user,ph)).fetchall()
    return rows or "Login failed"

print("--- Normal login ---")
print(vuln_login("alice", hashlib.sha256(b"alice123").hexdigest()))

print("\n--- SQLi Attack 1: Comment bypass (alice'--) ---")
r=vuln_login("alice'--","anything"); print(r)
if r!="Login failed": print("⚠ Auth bypassed without password!")

print("\n--- SQLi Attack 2: UNION data exfiltration ---")
r=vuln_login("' UNION SELECT id,data,'x','x' FROM secret_data--","x"); print(r)
if "TOP SECRET" in str(r): print("⚠ secret_data exfiltrated!")

print("\n--- Safe parameterised queries ---")
for u,p in [("alice'--","anything"),("' UNION SELECT 1,2,'x','x'--","x")]:
    print(f"  '{u}' → {safe_login(u,p)}")
print(f"  alice/alice123 → {safe_login('alice','alice123')}")

conn.close(); os.unlink(db)

print()
print("Countermeasures:")
for c in ["Parameterised queries / prepared statements","ORMs (SQLAlchemy, Django ORM)","Input validation + allow-listing","Principle of least privilege (PoLP)","WAF (Web Application Firewall)"]:
    print(f"  ✓ {c}")


## Summary

In [ ]:
print(f"{'Attack':<18} {'Layer':<18} {'Impact':<25} Countermeasure")
print('-'*85)
rows=[("ARP Spoofing","Layer 2 / LAN","MITM, sniff traffic","DAI, Static ARP, VPN/TLS"),
      ("DNS Poisoning","DNS (UDP/53)","Redirect → fake servers","DNSSEC, DoH, DoT"),
      ("SQL Injection","Application (DB)","Auth bypass, data theft","Parameterised queries")]
for a,l,i,c in rows:
    print(f"  {a:<16} {l:<18} {i:<25} {c}")
